## A1 – PBI Export Function (FactSales DataFrame)
Build a function that enriches clean data with Revenue, Profit, Margin, renames columns to PBI‑friendly names, and returns the FactSales DataFrame.

In [1]:
# A1: export_for_powerbi()
# Expects a DataFrame with Cancelled rows already removed.
# Adds financial columns and renames to Power BI‑ready names.

import pandas as pd
import os
from datetime import datetime

def export_for_powerbi(clean_df, output_path='outputs/pbi_techmart.xlsx'):
    """
    Returns a FactSales DataFrame with PBI‑friendly columns.
    The actual .xlsx saving happens later with DimCustomer + Metadata.
    """
    df = clean_df.copy()
    
    # Compute Revenue = Quantity * Unit_Price * (1 - Discount_Pct/100)
    df['Revenue_INR'] = df['Quantity'] * df['Unit_Price'] * (1 - df['Discount_Pct'] / 100)
    
    # Category margins (used for Profit_INR and Profit_Margin_Pct)
    margins = {
        'Electronics': 0.15,
        'Software': 0.65,
        'Accessories': 0.40,
        'Services': 0.55
    }
    df['Profit_INR'] = df['Revenue_INR'] * df['Category'].map(margins)
    df['Profit_Margin_Pct'] = df['Category'].map(margins) * 100

    # Rename Date -> Order_Date (PBI standard)
    df.rename(columns={'Date': 'Order_Date'}, inplace=True)

    # Reorder columns for clarity
    cols = ['Order_ID', 'Order_Date', 'Customer_ID', 'Customer_Name', 'Region',
            'City', 'Category', 'Product', 'Quantity', 'Unit_Price',
            'Discount_Pct', 'Revenue_INR', 'Profit_INR', 'Profit_Margin_Pct']
    
    return df[cols]

## A2 – DimCustomer Sheet
Aggregate one row per customer with Total_Orders, Total_Revenue_INR and Avg_Order_Value_INR.

In [2]:
# A2: DimCustomer – ONE row per customer (20 rows total)
# Group by Customer_ID and Customer_Name, then attach Region/City from the most frequent combination in the original data.

def build_dim_customer(fact_df, original_clean_df):
    """
    Returns a DimCustomer DataFrame with exactly one row per customer.
    Uses the most common Region and City for each customer from original_clean_df.
    """
    # 1. Find most frequent Region, City for each customer
    cust_geo = original_clean_df.groupby(['Customer_ID', 'Customer_Name'])[['Region', 'City']].agg(
        lambda x: x.value_counts().index[0]  # mode – most frequent value
    ).reset_index()
    
    # 2. Aggregate numeric metrics from FactSales (group by Customer_ID only)
    dim = fact_df.groupby(['Customer_ID', 'Customer_Name']).agg(
        Total_Orders=('Order_ID', 'count'),
        Total_Revenue_INR=('Revenue_INR', 'sum'),
        Avg_Order_Value_INR=('Revenue_INR', 'mean')
    ).reset_index()
    
    # 3. Merge the geographic mode back to the aggregated data
    dim = dim.merge(cust_geo, on=['Customer_ID', 'Customer_Name'], how='left')
    
    # 4. Reorder and sort
    dim = dim[['Customer_ID', 'Customer_Name', 'Region', 'City',
               'Total_Orders', 'Total_Revenue_INR', 'Avg_Order_Value_INR']]
    dim.sort_values('Total_Revenue_INR', ascending=False, inplace=True)
    
    return dim

## A3 – _metadata Sheet
Create a metadata table with script version, timestamp, row count, etc.

In [3]:
# A3: Metadata sheet – 6 fields as required

def create_metadata(fact_df, source_filename='TechMart_Raw_Data.xlsx'):
    """
    Creates a metadata DataFrame that will be written as the '_metadata' sheet.
    """
    meta = pd.DataFrame({
        'Field': [
            'Script Version',
            'Run Timestamp',
            'Source File',
            'Rows Exported (FactSales)',
            'Columns Exported',
            'Filters Applied'
        ],
        'Value': [
            '1.0',
            datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            source_filename,
            str(len(fact_df)),
            ', '.join(fact_df.columns),
            'Excluded: Status == Cancelled'
        ]
    })
    return meta

## A1+A2+A3 Combined – Write the Power BI .xlsx
Function that takes the cleaned DataFrame and exports a single .xlsx with 3 sheets.

In [4]:
# Write the full Power BI Excel file with FactSales, DimCustomer, _metadata

def write_pbi_excel(clean_df, output_path='outputs/pbi_techmart.xlsx'):
    # Build the three components
    fact = export_for_powerbi(clean_df)
    dim = build_dim_customer(fact, clean_df)   # ← fixed: pass clean_df as second arg
    meta = create_metadata(fact)
    
    os.makedirs('outputs', exist_ok=True)
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        fact.to_excel(writer, sheet_name='FactSales', index=False)
        dim.to_excel(writer, sheet_name='DimCustomer', index=False)
        meta.to_excel(writer, sheet_name='_metadata', index=False)
    
    print(f'✅ PBI export: {len(fact)} rows → {output_path}')
    print(f'✅ DimCustomer: {len(dim)} rows (unique customers)')
    return fact, dim

## B1 – Tableau CSV Export
Export the same FactSales as a CSV with UTF-8 BOM, YYYY-MM-DD dates, and no index.

In [5]:
# B1: Export to CSV for Tableau

def export_for_tableau(clean_df, output_dir='outputs/'):
    """
    Creates a Tableau‑ready CSV file.
    """
    # Reuse the same fact builder (already has financial columns)
    fact = export_for_powerbi(clean_df)
    
    os.makedirs(output_dir, exist_ok=True)
    csv_path = os.path.join(output_dir, 'tab_techmart.csv')
    
    # utf-8-sig for Windows compatibility, date format YYYY-MM-DD
    fact.to_csv(csv_path, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
    print(f'✅ Tableau CSV: {len(fact)} rows → {csv_path}')
    return fact

## B2 – Summary CSV for Tableau
Pre‑aggregate Region × Category for fast KPI loading.

In [6]:
# B2: Create summary CSV – one row per Region×Category

def create_summary_csv(fact_df, output_dir='outputs/'):
    """
    Aggregates revenue and margin by Region and Category.
    """
    summary = fact_df.groupby(['Region', 'Category']).agg(
        Total_Revenue=('Revenue_INR', 'sum'),
        Avg_Margin_Pct=('Profit_Margin_Pct', 'mean'),
        Order_Count=('Order_ID', 'count')
    ).reset_index()
    
    os.makedirs(output_dir, exist_ok=True)
    summary_path = os.path.join(output_dir, 'summary_for_tableau.csv')
    summary.to_csv(summary_path, index=False, encoding='utf-8-sig')
    print(f'✅ Tableau Summary CSV: {len(summary)} region×category combos → {summary_path}')
    return summary

## B3 – pantab / .hyper Attempt
Try to export to Tableau’s native .hyper format. If pantab is missing, log a clear explanation.

In [7]:
# B3: Attempt .hyper export using pantab

def attempt_hyper_export(fact_df, output_dir='outputs/'):
    """Tries to write fact_df to a .hyper extract."""
    hyper_path = os.path.join(output_dir, 'techmart_extract.hyper')
    try:
        import pantab
        pantab.frame_to_hyper(fact_df, hyper_path, table='Extract')
        print(f'✅ Hyper extract created → {hyper_path}')
    except ImportError:
        print('⚠ pantab not installed – .hyper export skipped.')
        print('   Reason: pantab requires the Tableau Hyper API (C library).')
        print('   Alternative: CSV works fine for <100K rows.')
        print('   When .hyper matters: >100K rows, Tableau Server scheduled refresh.')
    except Exception as e:
        print(f'⚠ Hyper export failed: {e}')

## C1 – Unified Pipeline Function
Read raw data, clean it, and call all export functions. Print a pipeline summary.

In [8]:
# C1: Full pipeline with cleaning and exports

def clean_raw_data(raw_df):
    """Exclude Cancelled, convert Date to datetime, warn if >10% cancelled."""
    cancel_pct = (raw_df['Status'] == 'Cancelled').mean() * 100
    if cancel_pct > 10:
        print(f'⚠ Warning: {cancel_pct:.1f}% of orders are Cancelled (>10% threshold)')
    df = raw_df[raw_df['Status'] != 'Cancelled'].copy()
    df['Date'] = pd.to_datetime(df['Date'])
    return df

def run_bi_export_pipeline(source_path):
    """
    Reads the source Excel, cleans, and creates all output files.
    Returns a dictionary with status and file details.
    """
    result = {'status': 'success', 'files_created': [], 'rows_processed': 0, 'errors': []}
    try:
        # 1. Read source
        raw = pd.read_excel(source_path, sheet_name='Raw Data', skiprows=1)
        
        # 2. Validate columns
        required = ['Order_ID', 'Date', 'Customer_ID', 'Category', 'Quantity', 'Unit_Price', 'Status']
        missing = [c for c in required if c not in raw.columns]
        if missing:
            raise ValueError(f'Missing columns: {missing}')
        
        # 3. Clean
        clean_df = clean_raw_data(raw)
        result['rows_processed'] = len(clean_df)
        
        # 4. Power BI Excel
        fact, _ = write_pbi_excel(clean_df)
        result['files_created'].append('outputs/pbi_techmart.xlsx')
        
        # 5. Tableau CSV + Summary
        fact_tab = export_for_tableau(clean_df)
        create_summary_csv(fact_tab)
        result['files_created'].extend(['outputs/tab_techmart.csv', 'outputs/summary_for_tableau.csv'])
        
        # 6. (Optional) .hyper
        attempt_hyper_export(fact_tab)
        
        # Final summary
        print(f'\n=== PIPELINE COMPLETE ===')
        print(f'Rows processed: {result["rows_processed"]}')
        print(f'Files created: {len(result["files_created"])}')
        print(f'Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
        
    except FileNotFoundError:
        result['status'] = 'error'
        result['errors'].append(f'File not found: {source_path}')
        print(f'ERROR: {source_path} not found')
    except ValueError as e:
        result['status'] = 'error'
        result['errors'].append(str(e))
        print(f'ERROR: {e}')
    except Exception as e:
        result['status'] = 'error'
        result['errors'].append(str(e))
        print(f'Unexpected error: {e}')
    
    return result

## C2 – Error Handling (already inside the pipeline)
The pipeline already handles `FileNotFoundError`, missing columns, and cancellation warnings.  
It always returns a structured result dict.

## D1 – Verification & NRA Insight
Run the pipeline, then inspect and print every output along with an actionable insight.

In [9]:
# D1: Verification – Print shapes, dtypes, and one NRA insight

# Run the pipeline (source file must be in the same folder)
result = run_bi_export_pipeline('Day91_Python_PBI_Tableau_Integration.xlsx')

# 1. FactSales
pbi_fact = pd.read_excel('outputs/pbi_techmart.xlsx', sheet_name='FactSales')
print('\n--- FactSales ---')
print('Shape:', pbi_fact.shape)
print('Key dtypes:')
print(pbi_fact.dtypes[['Order_Date', 'Revenue_INR', 'Profit_INR']])
print('\nFirst 3 rows:')
print(pbi_fact.head(3)[['Order_ID', 'Order_Date', 'Revenue_INR', 'Profit_INR']])

# 2. DimCustomer
pbi_dim = pd.read_excel('outputs/pbi_techmart.xlsx', sheet_name='DimCustomer')
print('\n--- DimCustomer ---')
print('Shape:', pbi_dim.shape)
print('Top 3 by Total Revenue:')
print(pbi_dim.nlargest(3, 'Total_Revenue_INR')[['Customer_Name', 'Total_Revenue_INR', 'Total_Orders']])

# 3. Tableau CSV
tab = pd.read_csv('outputs/tab_techmart.csv')
print('\n--- Tableau CSV ---')
print('Shape:', tab.shape)
print('Has Unnamed index?', 'Unnamed: 0' in tab.columns)

# 4. Metadata
meta = pd.read_excel('outputs/pbi_techmart.xlsx', sheet_name='_metadata')
print('\n--- Metadata ---')
print(meta.to_string(index=False))

# 5. NRA Insight (using real numbers)
total_rev = pbi_fact['Revenue_INR'].sum()
top_region = pbi_fact.groupby('Region')['Revenue_INR'].sum().idxmax()
top_region_rev = pbi_fact.groupby('Region')['Revenue_INR'].sum().max()
region_pct = (top_region_rev / total_rev) * 100

print('\n🔍 NRA INSIGHT')
print(f'₹{total_rev:,.0f} total revenue – {top_region} leads with ₹{top_region_rev:,.0f} ({region_pct:.1f}%)')
print(f'Action: increase sales support in {top_region} and bundle high‑margin Software/Services to further raise margin.')

⚠ Warning: 13.3% of orders are Cancelled (>10% threshold)
✅ PBI export: 130 rows → outputs/pbi_techmart.xlsx
✅ DimCustomer: 20 rows (unique customers)
✅ Tableau CSV: 130 rows → outputs/tab_techmart.csv
✅ Tableau Summary CSV: 16 region×category combos → outputs/summary_for_tableau.csv
⚠ pantab not installed – .hyper export skipped.
   Reason: pantab requires the Tableau Hyper API (C library).
   Alternative: CSV works fine for <100K rows.
   When .hyper matters: >100K rows, Tableau Server scheduled refresh.

=== PIPELINE COMPLETE ===
Rows processed: 130
Files created: 3
Timestamp: 2026-05-12 18:09:04

--- FactSales ---
Shape: (130, 14)
Key dtypes:
Order_Date     datetime64[ns]
Revenue_INR             int64
Profit_INR            float64
dtype: object

First 3 rows:
       Order_ID Order_Date  Revenue_INR  Profit_INR
0  ORD-20240001 2024-01-19        36000     19800.0
1  ORD-20240003 2024-04-06        13300      5320.0
2  ORD-20240004 2024-02-18       144000     79200.0

--- DimCustomer -